# 22회. 미니 프로젝트 ① — 헬륨 봄베 사용시간 계산기 다시 만들기

**목표:** 3단계에서 배운 **변수·딕셔너리·조건문·반복문·함수·문자열·날짜**를 모두 써서, 실제 업무 도구를 처음부터 만든다.

**원본 도구:** `2. 수질환경/5. 헬륨 봄베 사용시간 계산기` (`helium_calc.py`, `helium_calc.html`)
GC-MS·헤드스페이스에 쓰는 헬륨 봄베가 **출구 압력을 유지할 수 있는 시간**을 추정합니다.

### 진행 방법
문제를 **작은 함수 6개**로 나눠 하나씩 만들고 확인한 뒤, 마지막에 조립합니다. (17회 "한 함수 = 한 가지 일")

> ✅ **해답 노트북** — 연습 문제의 예시 답안이 채워져 있습니다.

## 0. 문제 이해 — 계산 원리 (원본 README)
온도가 일정하고 이상기체라고 가정하면 가스량은 절대압에 비례합니다.
```
사용 가능 가스량(L)  = 용적(L) × (현재 게이지압 − 출구압 − 여유압) ÷ 14.696 psi
사용 가능 시간(min) = 사용 가능 가스량(L) × 1000 ÷ 총 유량(mL/min)
```
**확인용 예:** 용적 47 L, 현재 600 psi, 출구 80 psi, 유량 50 mL/min
→ 가스량 ≈ **1663.0 L**, 시간 ≈ **23일 2시간 21분 (23.10일)**

> 원본 프로그램으로 확인한 값입니다. 우리가 만든 계산기도 **이 값이 나와야** 합니다. (20회: 기대 결과로 검산)

## 1단계. 단위 환산 — 딕셔너리 + 함수
압력은 psi, bar, MPa, kPa, atm 등 단위가 다양합니다. 모든 계산은 **psi로 바꿔서** 합니다.

In [ ]:
P_ATM_PSI = 14.696          # 1 atm = 14.696 psi

TO_PSI = {                  # 단위 → psi 환산계수
    "psi": 1.0,
    "bar": 14.5038,
    "mpa": 145.038,
    "kpa": 0.145038,
    "atm": 14.696,
}

### 🧪 1-1. `to_psi(value, unit)` 만들기
- `unit`은 대소문자를 가리지 않게 (`"Bar"`, `"BAR"`도 동작)
- 지원하지 않는 단위면 `ValueError`를 일으키기 → `raise ValueError(f"지원하지 않는 단위: {unit}")`
- 확인: `to_psi(5.5, "bar")` ≈ `79.77`, `to_psi(600, "psi")` = `600.0`

In [ ]:
def to_psi(value, unit):
    """압력값을 psi로 바꾼다."""
    unit = unit.lower()
    if unit not in TO_PSI:
        raise ValueError(f"지원하지 않는 단위: {unit}")
    return value * TO_PSI[unit]

print(round(to_psi(5.5, "bar"), 2))    # 79.77
print(to_psi(600, "PSI"))              # 600.0

## 2단계. 사용 가능 가스량
### 🧪 2-1. `usable_gas_l(volume_l, p_in_psi, cutoff_psi)` 만들기
- 공식: `volume_l × (p_in_psi − cutoff_psi) ÷ P_ATM_PSI`
- 현재 압력이 이미 cutoff보다 낮으면 **0**을 돌려주기 (음수 가스량은 없음) — 힌트: `max(값, 0)`
- 확인: `usable_gas_l(47, 600, 80)` ≈ `1663.0`, `usable_gas_l(47, 70, 80)` = `0`

In [ ]:
def usable_gas_l(volume_l, p_in_psi, cutoff_psi):
    """출구압(cutoff)까지 꺼낼 수 있는 가스량(L, 1 atm 환산)."""
    usable_psi = max(p_in_psi - cutoff_psi, 0)
    return volume_l * usable_psi / P_ATM_PSI

print(round(usable_gas_l(47, 600, 80), 1))   # 1663.0
print(usable_gas_l(47, 70, 80))              # 0.0

## 3단계. 사용 가능 시간
### 🧪 3-1. `minutes_left(gas_l, flows)` 만들기
- `flows`는 장비별 유량 **리스트** (mL/min). 합계로 계산
- 총 유량이 0 이하이면 `ValueError`
- 공식: `gas_l × 1000 ÷ 총 유량`
- 확인: `minutes_left(1663.0, [50])` ≈ `33260`, `minutes_left(1663.0, [30, 20])` 도 같은 값

In [ ]:
def minutes_left(gas_l, flows):
    """가스량(L)과 유량 목록(mL/min)으로 사용 가능 시간(분)을 구한다."""
    total = sum(flows)
    if total <= 0:
        raise ValueError("총 유량은 0보다 커야 합니다.")
    return gas_l * 1000 / total

print(round(minutes_left(1663.0, [50])))        # 33260
print(round(minutes_left(1663.0, [30, 20])))    # 33260

## 4단계. 보기 좋게 — 분을 "○일 ○시간 ○분"으로
### 🧪 4-1. `fmt_duration(minutes)` 만들기
- 반올림한 분을 하루(1440분), 한 시간(60분)으로 나누기 — `divmod(a, b)`는 `(몫, 나머지)`를 한 번에 돌려줍니다.
- 확인: `fmt_duration(33260.1)` → `"23일 2시간 20분"`, `fmt_duration(125)` → `"0일 2시간 5분"`

In [ ]:
def fmt_duration(minutes):
    """분 → 'D일 H시간 M분'"""
    days, rest = divmod(int(round(minutes)), 1440)
    hours, mins = divmod(rest, 60)
    return f"{days}일 {hours}시간 {mins}분"

print(fmt_duration(33260.1))   # 23일 2시간 20분
print(fmt_duration(125))       # 0일 2시간 5분

> 🤔 원본 결과는 `23일 2시간 21분`인데 여기서는 `20분`이 나왔습니다. 왜일까요?
> → 확인용 입력 `33260.1`은 **반올림된 가스량 1663.0**으로 계산한 값입니다. 반올림하지 않은 실제 가스량(1663.04…)으로 계산하면 21분이 됩니다. **중간 결과를 반올림하면 최종 결과가 달라질 수 있다** — 반올림은 **마지막에 한 번**만!

## 5단계. 언제 끝나나 — 날짜 (21회)
### 🧪 5-1. `end_time(start, minutes)` 만들기
- `start`(datetime)에 `minutes`분을 더한 시각
- 확인: `end_time(datetime(2025, 4, 8, 9, 0), 33260)` → `2025-05-01 11:20`

In [ ]:
from datetime import datetime, timedelta

def end_time(start, minutes):
    return start + timedelta(minutes=minutes)

print(end_time(datetime(2025, 4, 8, 9, 0), 33260).strftime("%Y-%m-%d %H:%M"))   # 2025-05-01 11:20

## 6단계. 주 단위 예상 압력표 — 반복문
1주마다 압력이 얼마나 떨어지는지 표로 보여 줍니다. 1분에 줄어드는 압력(psi)은
```
분당 압력 감소 = 총 유량(mL/min) ÷ 1000 × P_ATM_PSI ÷ 용적(L)
```
### 🧪 6-1. `weekly_table(volume_l, p_in_psi, flows, total_minutes)` 만들기
- 0주, 1주, 2주 … 의 예상 압력(psi)을 `(주, 압력)` 튜플의 리스트로 돌려주기
- 그 주의 시작 시점(`week × 7 × 24 × 60`분)이 `total_minutes`를 넘으면 멈추기 (`while`)
- 확인: 47 L, 600 psi, [50] → `[(0, 600.0), (1, 442.4), (2, 284.8), (3, 127.2)]` (소수 1자리)

In [ ]:
def weekly_table(volume_l, p_in_psi, flows, total_minutes):
    """주 단위 예상 게이지압(psi) 목록 [(주, 압력), ...]"""
    drop_per_min = sum(flows) / 1000 * P_ATM_PSI / volume_l
    table = []
    week = 0
    while week * 7 * 24 * 60 <= total_minutes:
        t = week * 7 * 24 * 60
        table.append((week, round(p_in_psi - drop_per_min * t, 1)))
        week += 1
    return table

print(weekly_table(47, 600, [50], 33260))

## 7단계. 조립 — 계산기 완성
위 함수들을 모아 `calculate(...)` 하나로 만들고, 결과를 보기 좋게 출력합니다.

In [ ]:
def calculate(volume_l, p_in, p_out, flows, unit="psi", margin=0.0, start=None):
    """헬륨 봄베 사용 가능 시간을 계산해 결과 딕셔너리로 돌려준다."""
    p_in_psi = to_psi(p_in, unit)
    cutoff_psi = to_psi(p_out + margin, unit)
    gas = usable_gas_l(volume_l, p_in_psi, cutoff_psi)
    minutes = minutes_left(gas, flows)
    start = start or datetime.now()
    return {
        "gas_l": gas,
        "minutes": minutes,
        "duration": fmt_duration(minutes),
        "days": minutes / 1440,
        "end": end_time(start, minutes),
        "weekly": [(w, p / TO_PSI[unit.lower()]) for w, p in weekly_table(volume_l, p_in_psi, flows, minutes)],
        "unit": unit,
    }

def print_result(r):
    print("===== 헬륨 봄베 사용 가능 시간 =====")
    print(f"사용 가능 가스량 : {r['gas_l']:.1f} L (1 atm 환산)")
    print(f"사용 가능 시간   : {r['duration']} ({r['days']:.2f} 일)")
    print(f"압력 유지 종료   : {r['end']:%Y-%m-%d %H:%M} 경")
    print("[주 단위 예상 게이지압]")
    for w, p in r["weekly"]:
        print(f"  +{w:>2}주  {p:8.1f} {r['unit']}")

r = calculate(47, 600, 80, [50], start=datetime(2025, 4, 8, 9, 0))
print_result(r)

## 8단계. 검증 — 원본과 비교
| 입력 | 원본 결과 | 내 결과 | 같은가 |
|---|---|---|---|
| 47 L, 600 psi, 출구 80, [50] | 1663.0 L, 23일 2시간 21분 | | |
| 47 L, 500 psi, 출구 80, 여유 20, [30, 20] | 1279.3 L, 17일 18시간 25분 | | |
| 47 L, 40 bar, 출구 5.5 bar, [50] | 1600.3 L, 22일 5시간 26분 | | |

### 🧪 8-1
위 세 가지 입력으로 `calculate`를 실행해 표를 채우세요.

In [ ]:
tests = [
    dict(volume_l=47, p_in=600, p_out=80, flows=[50]),
    dict(volume_l=47, p_in=500, p_out=80, flows=[30, 20], margin=20),
    dict(volume_l=47, p_in=40, p_out=5.5, flows=[50], unit="bar"),
]
for t in tests:
    r = calculate(**t, start=datetime(2025, 4, 8, 9, 0))
    print(f"{r['gas_l']:.1f} L, {r['duration']}")

---
## 마무리 — 무엇을 썼나?
| 배운 것 | 쓰인 곳 |
|---|---|
| 13 변수·계산 | 공식 옮기기, 상수 `P_ATM_PSI` |
| 14 딕셔너리·리스트 | `TO_PSI`, 유량 목록 |
| 15 조건문 | 단위 검사, 음수 가스량 처리 |
| 16 반복문 | 주 단위 표 (`while`) |
| 17 함수 | 6개 함수 + 조립 |
| 18 문자열 | `unit.lower()`, f-string 서식 |
| 20 오류 | `raise ValueError`, 기대 결과로 검산 |
| 21 날짜 | `timedelta`로 종료 시각 |

> 💬 이제 원본 `helium_calc.py`를 열어 보세요. **거의 모든 줄을 읽을 수 있을 겁니다.** 아직 안 배운 것은 `argparse`(명령줄 옵션) 정도입니다. 이게 3단계의 목표였던 "AI가 만든 코드를 읽을 수 있는 실력"입니다.

**과제:** `과제.md` — 노트북 계산기를 **`.py` 프로그램**으로 옮기기 (입력 받기)